In [1]:
from awsglue.context import GlueContext
from pyspark.context import SparkContext
from pyspark.sql.functions import *
from pyspark.sql.types import *

sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session



Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Trying to create a Glue session for the kernel.
Session Type: glueetl
Session ID: ad14f162-ada4-4940-8465-eac655d348dc
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session ad14f162-ada4-4940-8465-eac655d348dc to get into ready status...
Session ad14f162-ada4-4940-8465-eac655d348dc has been created.
ValueError: Cannot run multiple SparkContexts at once; existing SparkContext(app=GlueReplApp, master=jes) created by __init__ at /tmp/3582302106169943848:514 


In [9]:
# Read raw data from S3 
vitals_df = spark.read.option("header", True).csv(
    "s3://patient-health-monitoring/raw/vitals/"
)

patients_df = spark.read.option("header", True).csv(
    "s3://patient-health-monitoring/raw/patients/"
)

devices_df = spark.read.option("header", True).csv(
    "s3://patient-health-monitoring/raw/devices/"
)

In [3]:
print("Number of records in vitals table: ", vitals_df.count())

Number of records in vitals table:  6


In [6]:
# Remove Duplicates
vitals_df = vitals_df.dropDuplicates()
vitals_df.show()

+----------+-------------------+----------+--------------+------------+-----------+
|patient_id|          timestamp|heart_rate|blood_pressure|oxygen_level|temperature|
+----------+-------------------+----------+--------------+------------+-----------+
|     P1003|2025-01-10 08:10:00|        45|         90/60|          88|      101.2|
|     P1002|2025-01-10 08:05:00|        95|        140/90|          96|       99.1|
|     P1005|2025-01-10 08:20:00|       110|        150/95|        null|      100.1|
|     P1004|2025-01-10 08:15:00|      null|        130/85|          97|       98.4|
|     P1001|2025-01-10 08:00:00|        72|        120/80|          98|       98.6|
+----------+-------------------+----------+--------------+------------+-----------+


In [7]:
#Handle Null Values
vitals_df = vitals_df.fillna({
    "heart_rate": 0,
    "oxygen_level": 0
})
vitals_df.show()

+----------+-------------------+----------+--------------+------------+-----------+
|patient_id|          timestamp|heart_rate|blood_pressure|oxygen_level|temperature|
+----------+-------------------+----------+--------------+------------+-----------+
|     P1003|2025-01-10 08:10:00|        45|         90/60|          88|      101.2|
|     P1002|2025-01-10 08:05:00|        95|        140/90|          96|       99.1|
|     P1005|2025-01-10 08:20:00|       110|        150/95|           0|      100.1|
|     P1004|2025-01-10 08:15:00|         0|        130/85|          97|       98.4|
|     P1001|2025-01-10 08:00:00|        72|        120/80|          98|       98.6|
+----------+-------------------+----------+--------------+------------+-----------+


In [8]:
# Cast columns to correct data types
vitals_df = vitals_df \
    .withColumn("heart_rate", col("heart_rate").cast("integer")) \
    .withColumn("oxygen_level", col("oxygen_level").cast("integer")) \
    .withColumn("temperature", col("temperature").cast("double"))

In [10]:
# Health Status Rule
vitals_df = vitals_df.withColumn(
    "health_status",
    when(col("oxygen_level") < 90, "Critical")
    .when(col("temperature") > 100, "Fever")
    .when(col("heart_rate") > 100, "High Risk")
    .otherwise("Normal")
)
vitals_df.show()

+----------+-------------------+----------+--------------+------------+-----------+-------------+
|patient_id|          timestamp|heart_rate|blood_pressure|oxygen_level|temperature|health_status|
+----------+-------------------+----------+--------------+------------+-----------+-------------+
|     P1001|2025-01-10 08:00:00|        72|        120/80|          98|       98.6|       Normal|
|     P1002|2025-01-10 08:05:00|        95|        140/90|          96|       99.1|       Normal|
|     P1003|2025-01-10 08:10:00|        45|         90/60|          88|      101.2|     Critical|
|     P1004|2025-01-10 08:15:00|      null|        130/85|          97|       98.4|       Normal|
|     P1005|2025-01-10 08:20:00|       110|        150/95|        null|      100.1|    High Risk|
|     P1005|2025-01-10 08:20:00|       110|        150/95|        null|      100.1|    High Risk|
+----------+-------------------+----------+--------------+------------+-----------+-------------+


In [11]:
# Alert Flag
vitals_df = vitals_df.withColumn(
    "alert_flag",
    when(col("health_status") != "Normal", "Y")
    .otherwise("N")
)
vitals_df.show()


+----------+-------------------+----------+--------------+------------+-----------+-------------+----------+
|patient_id|          timestamp|heart_rate|blood_pressure|oxygen_level|temperature|health_status|alert_flag|
+----------+-------------------+----------+--------------+------------+-----------+-------------+----------+
|     P1001|2025-01-10 08:00:00|        72|        120/80|          98|       98.6|       Normal|         N|
|     P1002|2025-01-10 08:05:00|        95|        140/90|          96|       99.1|       Normal|         N|
|     P1003|2025-01-10 08:10:00|        45|         90/60|          88|      101.2|     Critical|         Y|
|     P1004|2025-01-10 08:15:00|      null|        130/85|          97|       98.4|       Normal|         N|
|     P1005|2025-01-10 08:20:00|       110|        150/95|        null|      100.1|    High Risk|         Y|
|     P1005|2025-01-10 08:20:00|       110|        150/95|        null|      100.1|    High Risk|         Y|
+----------+-------

In [13]:
# Monitoring Date and Hour
vitals_df = vitals_df \
    .withColumn("timestamp", to_timestamp("timestamp")) \
    .withColumn("monitoring_date", to_date("timestamp")) \
    .withColumn("monitoring_hour", hour("timestamp"))
vitals_df.show()


+----------+-------------------+----------+--------------+------------+-----------+-------------+----------+---------------+---------------+
|patient_id|          timestamp|heart_rate|blood_pressure|oxygen_level|temperature|health_status|alert_flag|monitoring_date|monitoring_hour|
+----------+-------------------+----------+--------------+------------+-----------+-------------+----------+---------------+---------------+
|     P1001|2025-01-10 08:00:00|        72|        120/80|          98|       98.6|       Normal|         N|     2025-01-10|              8|
|     P1002|2025-01-10 08:05:00|        95|        140/90|          96|       99.1|       Normal|         N|     2025-01-10|              8|
|     P1003|2025-01-10 08:10:00|        45|         90/60|          88|      101.2|     Critical|         Y|     2025-01-10|              8|
|     P1004|2025-01-10 08:15:00|      null|        130/85|          97|       98.4|       Normal|         N|     2025-01-10|              8|
|     P1005|2

In [15]:
# Join Datasets
final_df = vitals_df.join(
    patients_df,
    on="patient_id",
    how="inner"
).join(
    devices_df,
    on="patient_id",
    how="inner"
)
final_df.show()


+----------+-------------------+----------+--------------+------------+-----------+-------------+----------+---------------+---------------+------------+---+------+------+-------------+---------+------------+--------+-------------------+
|patient_id|          timestamp|heart_rate|blood_pressure|oxygen_level|temperature|health_status|alert_flag|monitoring_date|monitoring_hour|        name|age|gender|  city|      disease|device_id| device_type|  status|          last_sync|
+----------+-------------------+----------+--------------+------------+-----------+-------------+----------+---------------+---------------+------------+---+------+------+-------------+---------+------------+--------+-------------------+
|     P1001|2025-01-10 08:00:00|        72|        120/80|          98|       98.6|       Normal|         N|     2025-01-10|              8|Rahul Sharma| 45|  Male| Delhi|     Diabetes|     D101|  SmartWatch|  Active|2025-01-10 08:00:00|
|     P1002|2025-01-10 08:05:00|        95|     

In [16]:
final_report_df = final_df.select(
    "patient_id",
    col("name").alias("patient_name"),
    "disease",
    "heart_rate",
    "oxygen_level",
    "temperature",
    "health_status",
    "device_type",
    "monitoring_date"
)
final_report_df.show()

+----------+------------+-------------+----------+------------+-----------+-------------+------------+---------------+
|patient_id|patient_name|      disease|heart_rate|oxygen_level|temperature|health_status| device_type|monitoring_date|
+----------+------------+-------------+----------+------------+-----------+-------------+------------+---------------+
|     P1001|Rahul Sharma|     Diabetes|        72|          98|       98.6|       Normal|  SmartWatch|     2025-01-10|
|     P1002| Priya Verma| Hypertension|        95|          96|       99.1|       Normal|    Oximeter|     2025-01-10|
|     P1003|  Amit Singh|Heart Disease|        45|          88|      101.2|     Critical|HeartMonitor|     2025-01-10|
+----------+------------+-------------+----------+------------+-----------+-------------+------------+---------------+


In [ ]:
final_report_df.write \
    .mode("overwrite") \
    .parquet("s3://patient-health-monitoring/processed/reports/")